# A2 · MiMo-V2.6 (SWA + full · sink · partial RoPE · MoE) 순전파 검증

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_llm_dit/blob/main/notebooks/verify/A2_mimo_v26.ipynb)

원본: HF 원격 코드 `XiaomiMiMo/MiMo-V2.6-Flash-RL` 의 `modeling_mimo_v2.py` (`MiMoV2Model`, commit `dcd3fd8` 고정).
Flash / Pro 저장소의 모델 코드는 같은 파일입니다. 내 구현: `miniarch/models/mimo_v26.py`.
텍스트 backbone만 비교합니다 (vision/audio 인코더, MTP 층, DFlash draft 제외).

부품
- sliding window attention + full attention 하이브리드 (Flash 1:5, Pro 1:7), 층 종류마다 head 수·KV head·head_dim·RoPE theta가 다름
- q/k head_dim ≠ v head_dim (192 / 128), value에 상수 스케일(0.707) 곱함
- partial RoPE: head 앞쪽 33.4% 채널만 회전
- head별 attention sink (gpt-oss 방식, softmax 분모에만 추가). 원본 설정은 sliding 층에만 있음. QK-norm은 없음
- MoE: sigmoid 점수 + 보정 bias, 그룹 top-k → expert top-k, 첫 층만 dense

축소 config: 4층 = [full+dense, sliding+MoE, sliding+MoE, full+MoE], window 8, 시퀀스 40.
- full / sliding 층의 head_dim(24 / 30), KV head(1 / 2), theta(1e6 / 1e4)를 일부러 다르게 해서 설정이 뒤바뀌면 바로 드러나게 했습니다.
- rope 채널 = int(24×0.334)=8, int(30×0.334)=10 → 실제(192 → 64)처럼 앞쪽 일부만 회전합니다.
- expert 8개를 4그룹으로 나누고 2그룹만 고르게 해서 그룹 제한 경로를 씁니다 (실제 설정은 그룹 1개).
  routed_scaling_factor=2.5, eps 1e-5 도 기본값과 다르게 둬서 누락을 잡습니다.
- 원본 코드가 `torch.empty`로 만드는 sink·router 가중치·보정 bias는 0이 아닌 값으로 채웁니다.
- 실제 체크포인트와 같은 fused `qkv_proj`로 비교하고, 분리형(q/k/v_proj)은 자체 점검에서 한 번 더 비교합니다.

순서: 0. 환경 준비 → 1. 원본 구조 (HF 헤더) → 2. 원본 모델 동작 확인 → 3. 내 구현 == 원본 (층별 출력 비교) → 4. causal 누수 → 5. 단계 전용 자체 점검 → 6. 결과 저장

**런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 위에서부터 실행하세요. 다른 모델 노트북과 독립적으로 돌아갑니다.

## 0. 환경 준비

In [ ]:
import os
REPO_DIR = "/content/tmp_llm_dit"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/HisameOgasahara/tmp_llm_dit.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
if not os.environ.get("MINIARCH_SKIP_PIP"):   # 전체 실행 노트북이 한 번만 설치하고 넘길 때 건너뜀
    !pip -q install -U -r requirements-colab.txt
    # Colab 기본 torchao 가 최신 diffusers 와 버전이 안 맞아 import 가 깨짐 → 쓰지 않으므로 제거
    !pip -q uninstall -y torchao

In [ ]:
import sys, torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임 유형을 GPU로 바꾸세요)")
!git log -1 --format='repo commit: %h %s'

In [ ]:
from miniarch import verify as V
from miniarch.verify import stages
S = stages.get("A2")
dev = "cuda" if torch.cuda.is_available() else "cpu"
try:   # gated 모델 헤더/원격 코드용. Colab 왼쪽 🔑(보안 비밀)에 HF_TOKEN 등록.
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

CHECKS = []   # (이름, 통과, 오차) → 마지막에 저장
def report(name, ok, err=float("nan")):
    CHECKS.append({"check": name, "ok": bool(ok), "max_abs": float(err)})
    print(f"{'✅' if ok else '❌'} {name:<44} max_abs={err:.2e}")

## 원본 구조 (HF 헤더)

실제 체크포인트의 safetensors 헤더만 읽어 가중치 이름, shape, dtype을 봅니다 (다운로드 없음).
**구현 검증이 아니라 원본 구조 확인용**입니다. routed expert는 개수가 많아 숨깁니다.

In [ ]:
def peek(repo, pattern, subfolder="", limit=60):
    print(f"\n===== {repo}  /{pattern}/")
    try:
        cfg = V.fetch_config(repo, subfolder)
        cfg = cfg.get("text_config", cfg)
        keys = ["model_type", "hidden_size", "num_hidden_layers", "num_attention_heads", "num_key_value_heads", "head_dim"]
        print({k: cfg[k] for k in keys if k in cfg})
        h = V.read_model_header(repo, subfolder=subfolder, pattern=pattern)
        V.show({k: v for k, v in h.items() if ".experts." not in k}, limit)
    except Exception as e:
        print(f"⚠️ {type(e).__name__}: {e}")

peek('XiaomiMiMo/MiMo-V2.6-Flash-RL', '^model\\.layers\\.(0|1)\\.', subfolder='')

## 원본 모델 동작 확인

축소 config로 원본을 만들고 한 번 forward 합니다. 여기서 실패하면 라이브러리 버전 문제입니다 (내 구현과 무관).

In [ ]:
V.set_deterministic(0)
ref, inputs = S.ref()
ref = ref.to(dev).eval()
with torch.no_grad():
    out = ref(**{k: v.to(dev) if torch.is_tensor(v) else v for k, v in inputs.items()})
t = out[0] if isinstance(out, tuple) else getattr(out, "last_hidden_state", None)
t = t if t is not None else out.sample
print(f"{type(ref).__name__}  params={sum(p.numel() for p in ref.parameters()):,}  out={tuple(t.shape)}")

## 내 구현 == 원본

원본 가중치에 잡음을 넣고(초기값 0/1 때문에 복사 누락이 가려지지 않도록) 내 구현에 복사한 뒤, 층별 출력과 최종 출력을 비교합니다.

In [ ]:
row = V.cases.run_case(S.CASE)
print(row["status"]); print(row["detail"])
report("원본 동치 (" + S.CASE.reference + ")", row["status"].startswith("✅"))

## causal 누수

뒤쪽 토큰을 바꿨을 때 앞쪽 출력이 변하면 미래 정보가 새고 있다는 뜻입니다.

In [ ]:
import importlib
V.set_deterministic(0)
mine = importlib.import_module(S.CASE.mine_module).verify_pair(S.ref()[0])["mine"].to(dev).eval()
r = V.causal_leak_test(lambda x: mine(x).last_hidden_state, vocab_size=V.cases.VOCAB, seq_len=48, device=dev, atol=1e-4)
report("내 구현 " + r.name, r.ok, r.max_abs)

## 단계 전용 자체 점검

### attention map · sink · window 확인

- `attn_probs`: [B, H, T, T] (sink 칸 제외). 행 합 + sink 확률 = 1 이어야 합니다 (sink 없는 full 층은 행 합 = 1).
- 미래 칸과 sliding window 밖 칸의 확률은 정확히 0 이어야 합니다.

In [ ]:
from miniarch.models import mimo_v26 as M
ref, _ = S.ref()
m = M.verify_pair(ref)["mine"].to(dev).eval()
names = [n for n, _ in m.named_modules() if n.endswith(("attn_probs", "sink_probe"))]
T_ = 40
x = torch.randint(0, V.cases.VOCAB, (2, T_), device=dev)
with V.capture(m, names) as st, torch.no_grad():
    m(x)
i_, j_ = torch.arange(T_, device=dev)[:, None], torch.arange(T_, device=dev)[None, :]
for i in range(m.config.n_layer):
    kind = m.config.layer_types[i]
    p = st[f"layers.{i}.self_attn.attn_probs"]
    s = st.get(f"layers.{i}.self_attn.sink_probe", torch.zeros_like(p[..., 0]))
    err = (p.sum(-1) + s - 1).abs().max().item()
    outside = j_ > i_
    if kind == "sliding_attention":
        outside = outside | (i_ - j_ >= m.config.window)
    leak = p[..., outside].abs().max().item()
    print(f"layer {i} {kind:<18} probs={tuple(p.shape)}  행합+sink 오차={err:.1e}  마스크 밖 최대={leak:.1e}  평균 sink={s.mean():.3f}")
    report(f"layer {i} 확률 합(+sink)", err < 1e-5, err)
    report(f"layer {i} 마스크 밖 확률 0", leak == 0, leak)

### MoE 라우팅: 그룹 제한

토큰마다 고른 expert가 `topk_group`개 이하의 그룹 안에만 있어야 합니다. expert별 부하도 함께 봅니다.

In [ ]:
from miniarch.models import mimo_v26 as M
ref, _ = S.ref()
m = M.verify_pair(ref)["mine"].to(dev).eval()
names = [n for n, _ in m.named_modules() if n.endswith("route_probe")]
x = torch.randint(0, V.cases.VOCAB, (2, 40), device=dev)
with V.capture(m, names) as st, torch.no_grad():
    m(x)
c = m.config
per = c.n_experts // c.n_group
for n in names:
    idx = st[n]
    ng = max(len(set(row)) for row in (idx // per).tolist())
    load = torch.bincount(idx.flatten(), minlength=c.n_experts).tolist()
    print(f"{n:<28} 토큰당 그룹 수 최대={ng}  expert 부하={load}")
    report(f"{n} 그룹 수 <= {c.topk_group}", ng <= c.topk_group, float(ng))

### 분리형 q/k/v_proj 배치도 원본과 같은지

원본은 `attention_projection_layout`이 `"split"`이면 q/k/v를 따로 투영합니다 (계산은 같고 가중치 이름만 다름).

In [ ]:
from miniarch.models import mimo_v26 as M
V.set_deterministic(0)
ref_s, inp_s = S.ref(layout="split")
V.perturb_params(ref_s)
pair = M.verify_pair(ref_s)
mine_s = pair.pop("mine")
rep = V.check_equivalence(ref_s, mine_s, inp_s, atol=S.CASE.atol, device=dev, **pair)
print(rep.summary())
report("split q/k/v 배치 == 원본", rep.ok, max(r.max_abs for r in rep.results))

## 결과 저장

- `results/`에 점검 결과를 저장합니다.
- 노트북 출력까지 남기려면 **파일 → GitHub에 사본 저장** → 이 저장소의 `notebooks/verify/A2_mimo_v26.ipynb`.

In [ ]:
import datetime, pandas as pd
os.makedirs("results", exist_ok=True)
df = pd.DataFrame(CHECKS)
p = f"results/verify_A2_{datetime.datetime.now():%Y%m%d-%H%M}.csv"
df.to_csv(p, index=False)
print(df.to_string(index=False)); print("saved:", p)
print("전체:", "✅ 모두 통과" if df.ok.all() else "❌ 실패 있음")